# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
# paste your command here
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
# TODO: write a function, such as profile(name, table), that returns the four values in README
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.

def profile(name, table):
    dict = {
        'table': name,
        'columns': len(table.columns),
        'rows': len(table),
        'exact_duplicate_rows': table.duplicated().sum()
    }

    return dict

lst = []
for name in manifest['files']:
    lst.append(profile(name, raw[Path(name).stem]))

lst = pd.DataFrame(lst)
lst.to_csv(PROFILE_PATH, index=False)

## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

### 2.2 Stack the two visit files

In [4]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
visits_raw = None
visits_south = raw['visits_south'].copy(deep=True)
visits_south = visits_south.rename(columns={'wt_kg': 'weight_kg'})
visits_south['source_file'] = 'visits_south'

visits_north = raw['visits_north'].copy(deep=True)
visits_north['source_file'] = 'visits_north'

visits_raw = pd.concat([visits_north, visits_south], ignore_index=True)




### 2.3 Audit the raw tables

In [5]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.
issue_audit = pd.DataFrame({
    'patients: gender values that are not a code': (~raw['patients']['gender'].str.strip().isin(['F', 'M', 'X']).ne('')).sum(),
    'patients: empty gender': (raw['patients']['gender'].str.strip() == '').sum(),
    'patients: birth_year sentinel 9999': (raw['patients']['birth_year'] == '9999').sum(),
    'patients: site codes needing normalization': (raw['sites']['site_code'] != raw['sites']['site_code'].str.strip().str.upper().str.replace('-', '')).sum(),
    'sites: rows with a repeated site_code': raw['sites'].duplicated(subset=['site_code'], keep=False),
    'visits: visit labels that are not baseline or week12': ((visits_raw['visit'] != 'baseline') | (visits_raw['visit'] != 'week12')).sum(),
    'visits: sbp sentinel 999': (visits_raw['sbp_mmhg'] == '999').sum(),
    'visits: sbp outside 70 through 250': ((visits_raw['sbp_mmhg'] != '999') & ((pd.to_numeric(visits_raw['sbp_mmhg']) < 70) | (pd.to_numeric(visits_raw['sbp_mmhg']) > 250))).sum(),
    'visits: date parse failures': (pd.to_datetime(visits_raw['visit_date'], format='%Y-%m-%d', errors="coerce")).isna().sum(),
    'visits: rows in repeated patient_id and visit pairs': visits_raw[visits_raw['visit'].isin(['baseline', 'week12'])].duplicated(subset=['patient_id', 'visit'], keep=False) #TODO
})

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [6]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
patients_clean = raw['patients'].copy(deep=True)
gender_map = {
    'f': 'F',
    'female': 'F',
    'woman': 'F',
    'male': 'M',
    'm': 'M',
    'man': 'M',
    'x':'X',
    'nb': 'X',
    'nonbinary': 'X',
    'non-binary': 'X'
}

patients_clean = patients_clean.drop_duplicates()
# print(patients_clean['gender'].value_counts())
patients_clean['gender'] = patients_clean['gender'].str.strip().str.lower().map(gender_map).astype("category")
patients_clean['birth_year'] = patients_clean['birth_year'].replace('9999', None)
patients_clean['site_code'] = patients_clean['site_code'].str.upper().str.strip().str.replace("-", "").str.replace(" ", "")
patients_clean.reset_index(drop=True)
print(patients_clean)



      patient_id gender birth_year site_code
0         P00001      F       1966       S02
1         P00002      F       <NA>       N01
2         P00003      F       1970       N01
3         P00004      X       <NA>       N01
4         P00005      M       1947       N01
...          ...    ...        ...       ...
29995     P29996      F       1967       S02
29996     P29997      F       1970       N01
29997     P29998    NaN       1967       N01
29998     P29999      M       1956       S02
29999     P30000      M       1971       S02

[30000 rows x 4 columns]


In [7]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.

patients_clean['gender'].cat.categories

patients_df = pd.DataFrame({
    'patients': len(patients_clean),
    'birth_year missing': patients_clean['birth_year'].isna().sum(),
    'birth_year min': patients_clean['birth_year'].min(),
    'birth_year max': patients_clean['birth_year'].max(),
    'site_code values': patients_clean['site_code'].nunique(),
    'patients at N01': (patients_clean['site_code'] == 'N01').sum(),
    'patients at S02': (patients_clean['site_code'] == 'S02').sum(),
    'patients at E04': (patients_clean['site_code'] == 'E04').sum()
}, index=['Values'])

print(patients_df)

patients_df.to_csv(PATIENTS_SUMMARY_PATH)
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

        patients  birth_year missing birth_year min birth_year max  \
Values     30000                 371           1941           1996   

        site_code values  patients at N01  patients at S02  patients at E04  
Values                 3            13739            15706              555  


### 3.2 Clean the visits

In [28]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
visits_clean = visits_raw.copy(deep=True)
visits_clean = visits_clean.drop_duplicates()
# print(visits_clean['visit'].value_counts())
visit_map = {
    'bl' : 'baseline',
    'wk12': 'week12',
    'baseline' : 'baseline',
    'week12': 'week12'
}
visits_clean['visit'] = visits_clean['visit'].str.strip().str.replace(' ', '').str.lower().map(visit_map)
visits_clean['sbp_mmhg'] = pd.to_numeric(visits_clean['sbp_mmhg'], errors='coerce')
visits_clean['sbp_mmhg'] = visits_clean['sbp_mmhg'].mask((visits_clean['sbp_mmhg'] < 70) | (visits_clean['sbp_mmhg'] > 250) | (visits_clean['sbp_mmhg'].isna()), other=999)
visits_clean['weight_kg'] = pd.to_numeric(visits_clean['weight_kg'], errors='coerce')
visits_clean['visit_date'] = pd.to_datetime(visits_clean['visit_date'], format='%Y-%m-%d', errors="coerce")
visits_clean.duplicated(subset=['patient_id', 'visit'], keep=False)
visits_clean = visits_clean.sort_values('visit_time').drop_duplicates(subset=['patient_id', 'visit'], keep='last')
visits_clean = visits_clean.sort_values(by= ['patient_id', 'visit']).reset_index(drop=True)

# visits_clean["visit"].value_counts(dropna=False)


In [ ]:
# TODO: step 6, show the repeated patient_id and visit pairs, then keep the later reading.
visits_clean.duplicated(subset=['patient_id', 'visit'], keep=False)



In [43]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).

visits_clean = visits_clean.sort_values('visit_time').drop_duplicates(subset=['patient_id', 'visit'], keep='last')
visits_clean = visits_clean.sort_values(by= ['patient_id', 'visit']).reset_index(drop=True)
visits_clean["visit"].value_counts(dropna=False)

visits_summary = pd.DataFrame([
    ['visits', len(visits_clean['visit'])],
    ['baseline visits', len(visits_clean[visits_clean['visit'] == 'baseline'])],
    ['week12 visits', len(visits_clean[visits_clean['visit'] == 'week12'])],
    ['visit_date missing', (visits_clean['visit_date'].isna()).sum()],
    ['visit_date min', min(visits_clean['visit_date'])],
    ['visit_date max', max(visits_clean['visit_date'])],
    ['sbp_mmhg missing', (visits_clean['sbp_mmhg'].isna()).sum()],
    ['sbp_mmhg min', visits_clean['sbp_mmhg'].min()],
    ['sbp_mmhg max', visits_clean['sbp_mmhg'].max()],
    ['weight_kg missing', (visits_clean['weight_kg'].isna()).sum()],
    ['weight_kg mean', round(visits_clean['weight_kg'].mean(), 2)],
    ['visits from visits_north.csv', len(visits_clean[visits_clean['source_file'] == 'visits_north'])],
    ['visits from visits_south.csv', len(visits_clean[visits_clean['source_file'] == 'visits_south'])]
], columns=['metric', 'value'])

visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)



### 3.3 Record the decisions

In [11]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table = pd.DataFrame(columns=["table", "issue", "decision", "reason"])
decision_table

,table,issue,decision,reason


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [13]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [14]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [15]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [17]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [19]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [20]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.